<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row represents one pseudonymized content page at the March 31, 2026 decision point.** The underlying warehouse table is daily, with one row per `report_date × client_hash_id × content_hash_id`. I aggregate those daily observations to one row per `client_hash_id × content_hash_id` for the decision frame.

**Feature window:** March 1–31, 2026. I use the March history available up to the March 31 decision cutoff to construct the candidate features.

**Label window:** April 1–30, 2026. I use the following month only to measure the observed outcome after the decision point.

My lane is **Refresh / Content Opportunity Scoring**. The practical decision is which pages should be reviewed first for refresh or related content action. The observed change in clicks is used as a **proxy/evaluation signal**, not as proof that a refresh will cause recovery.

The warehouse source is `fact_content_daily_performance`. I use the daily performance table for the March feature window and aggregate it to the page-level decision grain rather than treating each daily row as an independent page.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

I use five candidate features for the initial ranking frame:

1. **`gsc_impressions_march`** — total Google Search Console impressions observed during March 2026.
2. **`gsc_clicks_march`** — total Google Search Console clicks observed during March 2026.
3. **`gsc_ctr_march`** — March clicks divided by March impressions.
4. **`gsc_avg_position_march`** — impression-weighted average Google Search Console position during March.
5. **`ga4_sessions_march`** — total GA4 sessions observed during March where GA4 data is available.

### Label / proxy

The evaluation proxy is **negative click movement from March to April**. I define `decline_label = 1` when April clicks are lower than March clicks, and `0` otherwise. This is an observed outcome used to evaluate the ranking frame; it is not a causal claim that a page requires a refresh or that refreshing it will cause recovery.

### Context

The following fields provide context rather than model features:

* `client_hash_id`
* `content_hash_id`
* `report_date`
* data-availability indicators
* content metadata used only to understand the slice

### Excluded

I exclude `april_clicks` from the feature set because it belongs to the future outcome window and is therefore unavailable at the March 31 decision point.

I also exclude product decision fields such as `health_score`, `priority_score`, `action_type`, or refresh flags. These are downstream product decisions rather than independent observable signals.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

if HF_TOKEN:
    print("HF_TOKEN loaded successfully.")
else:
    print("HF_TOKEN not found.")

HF_TOKEN loaded successfully.


In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [4]:
FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
    LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


### Query 1 — Grain verification

In [5]:
q1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT
            CAST(report_date AS VARCHAR)
            || '|' ||
            client_hash_id
            || '|' ||
            content_hash_id
        ) AS distinct_grain_keys
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
""").df()

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


**Interpretation:** The March source contains 9,841,378 rows and 9,841,378 distinct `report_date × client_hash_id × content_hash_id` keys. This supports the stated daily source grain: there is one observed row per date, client, and content item in this March slice.


#Query 2 — Row count + date span

In [6]:
q2 = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
""").df()

q2

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


**Interpretation:** The March feature source covers the complete stated feature window, from March 1 through March 31, 2026.


#Query 3 — Availability

In [7]:
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
""").df()

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows
0,9841378,3611061


**Interpretation:** GSC data is available for 3,611,061 of the 9,841,378 March rows, or approximately 36.69%. I therefore restrict GSC-derived features to observations where `gsc_data_available IS TRUE` rather than treating unavailable GSC data as zero performance.


In [8]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FACT}')
""").df()

schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [9]:
feature_sql = f"""
WITH march AS (
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions_march,

    SUM(gsc_clicks) AS gsc_clicks_march,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_ctr_march,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_avg_position_march,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE NULL
        END
    ) AS ga4_sessions_march

FROM march

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(feature_sql).df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,2.298246,NaN
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584,4.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,9.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,3.0


### Feature availability

The five candidate features are constructed from observations available during the March 1–31, 2026 feature window.

GSC-derived features are restricted to rows where `gsc_data_available IS TRUE`.

`ga4_sessions_march` remains missing when GA4 is unavailable. I do not replace missing analytics with zero because unavailable tracking does not mean that zero sessions occurred.

All five candidate features therefore use information that could have been observed by the March 31, 2026 decision cutoff.


In [ ]:
print("Feature frame shape:", features.shape)
print()
print("Missing values:")
print(features.isna().sum())

Feature frame shape: (176738, 7)

Missing values:
client_hash_id               0
content_hash_id              0
gsc_impressions_30d          0
gsc_clicks_30d               0
gsc_ctr_30d                  0
gsc_avg_position_30d         0
ga4_sessions_30d        112882
dtype: int64


### Feature-quality check

The resulting feature frame contains 176,738 client-content observations and five candidate features.

The four GSC-derived features have no missing values after filtering to `gsc_data_available IS TRUE`. `ga4_sessions_march` is missing for 112,882 observations, or approximately 63.87% of the feature frame.

I keep these GA4 values missing rather than converting them to zero because unavailable analytics tracking does not imply zero sessions. Any later model using this feature must therefore handle missingness explicitly.


### Query 4 — Verify the future outcome window

In [10]:
outcome_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_clicks,
    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS decline_label

FROM march m

INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE a.april_clicks IS NOT NULL
"""

outcomes = con.sql(outcome_sql).df()

outcomes.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_clicks,april_clicks,decline_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,8.0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,2.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,4.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,8.0,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,0.0,1


### Temporal outcome definition

The feature window ends on March 31, 2026, which is the decision cutoff. The outcome window is April 1–30, 2026.

`decline_label = 1` when a page's April GSC clicks are lower than its March GSC clicks; otherwise it is `0`.

April data is used only to construct the future outcome. It is not part of the feature set available at the March decision point.


In [ ]:
print("Outcome frame shape:", outcomes.shape)
print()
print(outcomes["decline_label"].value_counts())

Outcome frame shape: (158549, 5)

decline_label
0    114244
1     44305
Name: count, dtype: int64


### Query 5 — Verify the final feature + outcome frame

In [11]:
contract_df = features.merge(
    outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Contract dataframe shape:", contract_df.shape)
contract_df.head()

Contract dataframe shape: (158549, 10)


,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march,march_clicks,april_clicks,decline_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,NaN,2.0,2.0,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,2.298246,NaN,0.0,0.0,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584,4.0,0.0,0.0,0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,9.0,6.0,30.0,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,3.0,16.0,6.0,1


In [12]:
print(contract_df["decline_label"].value_counts(normalize=True))

decline_label
0    0.72056
1    0.27944
Name: proportion, dtype: float64


In [13]:
required_columns = [
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions_march",
    "gsc_clicks_march",
    "gsc_ctr_march",
    "gsc_avg_position_march",
    "ga4_sessions_march",
    "march_clicks",
    "april_clicks",
    "decline_label",
]

print("Required columns present:")
print(set(required_columns).issubset(contract_df.columns))

print("\nContract dataframe shape:", contract_df.shape)

print("\nMissing values in outcome:")
print(contract_df[["march_clicks", "april_clicks", "decline_label"]].isna().sum())

Required columns present:
True

Contract dataframe shape: (158549, 10)

Missing values in outcome:
march_clicks     0
april_clicks     0
decline_label    0
dtype: int64


**Interpretation:** The final contract frame joins the March feature observations to the subsequent April outcome using `client_hash_id × content_hash_id`. The April outcome is retained for evaluation, while April information is excluded from the candidate feature set.


### Leakage boundary

The model features are restricted to information available by the March 31, 2026 decision cutoff. April clicks are used only to construct the future `decline_label` and must not be included as an input feature.

A deliberate leakage experiment comparing leaked and honest models belongs in the separate feature-leakage check.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation

A major limitation of this slice is uneven analytics availability. GSC-derived features are constructed only where GSC data is available, while GA4 sessions remain unavailable for 112,882 of the 176,738 feature rows (approximately 63.87%).

Therefore, analytics-based signals are not uniformly observed across the feature frame. Missing GA4 data should not be interpreted as zero traffic, and any later model using `ga4_sessions_march` must handle missingness explicitly.

The observed March-to-April decline signal also describes what happened after the decision point; it does not establish that refreshing a page would cause its performance to recover.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.